<a href="https://colab.research.google.com/github/feixh/colab-notebooks/blob/main/class_to_image.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Class label conditioned image generation


- Dataset: CIFAR10
- Architecture: A small DiT operating in the pixel space
- GPU needed: T4 (15 GB vRAM) should be good
  - For debugging, a CPU instance should be good, but make sure to reduce the batch size when debugging.

## Install dependencies

In [ ]:
!pip install loguru einsum jaxtyping ipytest

## Workspace setup

- Mount Google Drive so that we have persistent storage
- Set up a workspace directory, etc.

In [ ]:
from torch.utils import data
import json
import pickle
import tqdm
import random
from pathlib import Path
from loguru import logger
import wandb
import ipytest
ipytest.autoconfig(addopts=['-W ignore::pytest.PytestAssertRewriteWarning'])

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

project_dir = Path("/content/drive/My Drive/Colab Notebooks/ml_coding_prep")
workspace = project_dir / "workspace_class_to_image"

if workspace.exists():
  logger.warning(f"Workspace already exists @ {workspace} -- Make sure you actually want to use the same workspace!!!")
else:
  workspace.mkdir(parents=True, exist_ok=False)
  logger.info(f"workspace created @ {workspace}")

data_dir = workspace / "data"
if data_dir.exists():
  logger.warning(f"Data directory already exists @ {data_dir} -- Make sure you actually want to use the same workspace!!!")
else:
  data_dir.mkdir(parents=True, exist_ok=False)
  logger.info(f"Data directory created @ {data_dir}")



In [ ]:
import wandb
from google.colab import userdata

# Retrieve the wandb API key from Colab Secrets
try:
  wandb_api_key = userdata.get('WANDB_API_KEY')
  # Log in to wandb programmatically
  wandb.login(key=wandb_api_key)
  logger.info("Successfully logged into W&B!")
except userdata.SecretNotFoundError:
  logger.error("WANDB_API_KEY not found in Colab Secrets. Please add it to the Secrets manager.")


## Data preparation

In [ ]:
from typing import List, Dict, Tuple
from jaxtyping import Float, Int

import torch
from torch import Tensor
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader


def my_collate_fn(batch: List[Tuple[Float[Tensor, "C H W"], int]]) -> Dict[str, Tensor]:
  images = []
  labels = []
  for image, label in batch:
    images.append(image)
    labels.append(label)

  images = torch.stack(images)  # [B C H W]
  labels = torch.tensor(labels, dtype=torch.long) # [B]
  out_batch = {"images": images, "labels": labels}
  return out_batch

# Define transformations for the CIFAR10 dataset
# For training, include data augmentation (random horizontal flip and random crop)
# For both, convert to tensor and normalize
transform_train = transforms.Compose([
    # transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

# For testing, only apply normalization.
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

# Load the CIFAR10 training dataset
trainset = torchvision.datasets.CIFAR10(
    root=data_dir, train=True, download=True, transform=transform_train)

# Create a DataLoader for the training set
trainloader = DataLoader(
    trainset,
    batch_size=128,
    shuffle=True,
    num_workers=2,
    collate_fn=my_collate_fn)

# Load the CIFAR10 test dataset
testset = torchvision.datasets.CIFAR10(
    root=data_dir,
    train=False,
    download=True,
    transform=transform_test)

# Create a DataLoader for the test set
testloader = DataLoader(
    testset,
    batch_size=100,
    shuffle=False,
    num_workers=2)

# Define the classes for CIFAR10
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

print(f"Training set size: {len(trainset)}")
print(f"Test set size: {len(testset)}")
print(f"Number of classes: {len(classes)}")

In [ ]:
%%ipytest

def test_dataloader():
  for batch in trainloader:
    assert "images" in batch
    assert "labels" in batch
    assert batch["images"].shape == (128, 3, 32, 32)
    assert batch["labels"].shape == (128,)
    break

## Model definition

Implement the simplest possible thing:
1. Class label -> embedding
2. Timestep -> embedding
3. Pachification -> reshape and linear mapping
4. Concatenate all the embeddings + position embedding
5. Transformer layers!
6. Final layer: pre-norm followed by a linear mapping followed by reshape

### Definition of layers


In [ ]:
from torch import nn
import math

import torch.nn.functional as F
from einops import rearrange, einsum, repeat
import torch

def get_sinusoidal_embeddings(n_seq, d_model):
    """
    Args:
        n_seq: The maximum sequence length.
        d_model: The hidden dimension of the transformer.
    Returns:
        A tensor of shape [n_seq, d_model]
    """
    # 1. Create a position vector: [n_seq, 1]
    position = torch.arange(n_seq).unsqueeze(1)

    # 2. Create the division term (the "frequencies"): [d_model / 2]
    # We only compute it for half the dimensions because we pair sin and cos
    div_term = torch.exp(torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model))

    # 3. Create the empty embedding matrix
    pe = torch.zeros(n_seq, d_model)

    # 4. Fill with sin and cos
    # [n_seq, 1] * [1, d_model/2] -> [n_seq, d_model/2] (via broadcasting)
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)

    return pe

def get_sinusoidal_embeddings_2d(height: int, width: int, d_model: int):
  assert d_model % 2 == 0
  half_dim = d_model // 2
  _feat_h = get_sinusoidal_embeddings(height, half_dim) # [height, half_dim]
  _feat_w = get_sinusoidal_embeddings(width, half_dim)  # [width, half_dim]
  # einops' repeat op handles both interleaved and non-interleaved repeat
  _feat_h = repeat(_feat_h, "h d -> (h w) d", w=width)  # interleaved repeat (new index w appears behind h)
  _feat_w = repeat(_feat_w, "w d -> (h w) d", h=height) # whole sequence repeat (new index h appears before w)
  _feat = torch.cat([_feat_h, _feat_w], dim=-1) # [height * width, d_model]
  return _feat


class LinearPachification(nn.Module):
  def __init__(self, dim: int, patch_size: int, channels: int):
    super().__init__()
    self.patch_size = patch_size
    # Note, we set the bias to True here to compensate for the non-zero offset in the pixels
    self.linear = nn.Linear(in_features=channels * patch_size ** 2,
                            out_features=dim,
                            bias=True)

  def forward(self, x: Float[Tensor, "B C H W"]) -> Float[Tensor, "B S D"]:
    b, c, h, w = x.shape
    assert h % self.patch_size == 0
    assert w % self.patch_size == 0
    x = rearrange(x, "b c (ph h) (pw w) -> b (h w) (ph pw c)",
                  ph=self.patch_size,
                  pw=self.patch_size)  # B S D
    x = self.linear(x)
    return x


class LearnablePositionEmbedding(nn.Module):
  def __init__(self, dim: int, max_seq_len: int):
    super().__init__()
    self.dim = dim
    self.max_seq_len = max_seq_len

    self.pos_emb = nn.Embedding(max_seq_len, dim)

  def forward(self, x: Float[Tensor, "B S D"], idx: Int[Tensor, "B S"]) -> Float[Tensor, "B S D"]:
    out = x + self.pos_emb(idx)
    return out

class AdaLN(nn.Module):
  def __init__(self, dim: int):
    super().__init__()
    self.norm = nn.LayerNorm(dim, elementwise_affine=False)

    self.adaln_linear = nn.Linear(dim, 3 * dim, bias=False)

    self.reset_parameters()

  def reset_parameters(self):
    nn.init.zeros_(self.adaln_linear.weight)
    if self.adaln_linear.bias is not None:
      nn.init.zeros_(self.adaln_linear.bias)

  def forward(self,
              x: Float[Tensor, "B S D"],
              cond: Float[Tensor, "B D"] = None,
              ) -> Tuple[Float[Tensor, "B S D"], Float[Tensor, "B 1 D"]]:
    x = self.norm(x)
    scale, shift, gate = torch.chunk(self.adaln_linear(F.silu(cond)), chunks=3, dim=-1) # [B D]
    scale, shift, gate = [each.unsqueeze(1) for each in (scale, shift, gate)] # [B 1 D]
    x = (1 + scale) * x + shift
    return x, gate


class FinalLayer(nn.Module):
  def __init__(self, *, dim: int, patch_size: int, out_channels: int, out_height: int, out_width: int):
    super().__init__()
    self.dim = dim
    self.patch_size = patch_size
    self.height = out_height
    self.width = out_width
    assert out_height % patch_size == 0
    assert out_width % patch_size == 0

    self.adaln = AdaLN(dim)
    self.linear = nn.Linear(
        in_features=dim,
        out_features=out_channels * patch_size ** 2,
        bias=True,  # set bias to true to compensate for the average color
      )

    self.reset_parameters()

  def reset_parameters(self):
    # zero-initialize the final linear layer so that at the beginning of training,
    # it predicts zero velocity -- this can help stablize the training.
    nn.init.zeros_(self.linear.weight)
    if self.linear.bias is not None:
      nn.init.zeros_(self.linear.bias)
    logger.info("FinalLayer's linear component is zero-initialized!")

  def forward(self, x: Float[Tensor, "B S D"], cond: Float[Tensor, "B D"]) -> Float[Tensor, "B C H W"]:
    x, _ = self.adaln(x, cond)  # ignore the gate
    x = self.linear(x) # B S D
    x = rearrange(x, "b (h w) (ph pw c) -> b c (ph h) (pw w)",
                  ph=self.patch_size, pw=self.patch_size,
                  h=self.height // self.patch_size,
                  w=self.width // self.patch_size)
    return x



class Attention(nn.Module):
  def __init__(self,
               dim: int,
               max_seq_len: int = 128):
    super().__init__()
    self.dim = dim
    self.max_seq_len = max_seq_len

    self.qkv = nn.Linear(dim, 3 * dim, bias=False)
    # (Optionally) we can dropout some attention
    self.proj = nn.Linear(dim, dim, bias=False)


  def forward(self, x: Float[Tensor, "B S D"]) -> Float[Tensor, "B S D"]:
    """
    x: shape [B, S, D]
    """
    seq_len = x.shape[1]
    assert seq_len <= self.max_seq_len, f"Sequence length {seq_len} is greater than max sequence length {self.max_seq_len}"

    qkv = self.qkv(x) # [B, S, 3*D]
    q, k, v = qkv.chunk(3, dim=-1)  # each has shape [B, S, D]
    dot_product = einsum(q, k, "b s_q d, b s_k d -> b s_q s_k") # [B, Sq, Sk] where Sq == Sk
    scaled_dot_product = dot_product / (self.dim ** 0.5)  # normalize by hidden dim

    attention = scaled_dot_product.softmax(dim=-1)
    out = einsum(attention, v, "b s_q s_k, b s_k d -> b s_q d")
    out = self.proj(out)  # final projection

    return out

class MultiHeadAttention(nn.Module):
  def __init__(self,
               dim: int,
               heads: int,
               max_seq_len: int = 128):
    super().__init__()
    assert dim % heads == 0, "dim must be divisible by heads"
    self.dim = dim
    self.heads = heads
    self.head_dim = dim // heads

    self.max_seq_len = max_seq_len
    self.qkv = nn.Linear(dim, 3 * dim, bias=False)
    self.proj = nn.Linear(dim, dim, bias=False)


  def forward(self, x: Float[Tensor, "B S D"]) -> Float[Tensor, "B S D"]:
    seq_len = x.shape[1]
    assert seq_len <= self.max_seq_len, f"Sequence length {seq_len} is greater than max sequence length {self.max_seq_len}"

    qkv = self.qkv(x)
    q, k, v = qkv.chunk(3, dim=-1)  # B S D
    # split along head dimension
    # Corrected pattern: (h d) to decompose 'dim' into 'heads' and 'head_dim'
    q, k, v = [rearrange(each,
                         "b s (h d) -> b h s d", # Changed 'hd' to '(h d)'
                         h=self.heads,
                         d=self.head_dim) # Pass explicit sizes for h and d
              for each in [q, k, v]]
    dot_product = einsum(q, k, "b h s_q d, b h s_k d -> b h s_q s_k")
    scaled_dot_product = dot_product / (self.head_dim ** 0.5)
    attention = scaled_dot_product.softmax(dim=-1)
    out = einsum(attention, v, "b h s_q s_k, b h s_k d -> b h s_q d")
    out = rearrange(out, "b h s d -> b s (h d)", h=self.heads, d=self.head_dim) # Changed 'hd' to '(h d)'
    out = self.proj(out)
    return out


class Mlp(nn.Module):
  def __init__(self, dim: int):
    super().__init__()
    # Note, the MLP's linear layers do not have biases as we
    # are just expanding and contracting the features.
    self.linear1 = nn.Linear(dim, 4 * dim, bias=False)
    self.linear2 = nn.Linear(4 * dim, dim, bias=False)

  def forward(self, x: Float[Tensor, "B S D"]) -> Float[Tensor, "B S D"]:
    return self.linear2(F.gelu(self.linear1(x)))


class TransformerLayer(nn.Module):
  def __init__(self,
               dim: int,
               heads: int,
               max_seq_len: int = 128):
    super().__init__()
    self.dim = dim
    self.heads = heads

    self.norm1 = nn.RMSNorm(dim)
    self.attn = MultiHeadAttention(dim=dim,
                                   heads=heads,
                                   max_seq_len=max_seq_len)
    self.norm2 = nn.RMSNorm(dim)
    self.mlp = Mlp(dim=dim)

  def forward(self, x: Float[Tensor, "B S D"]):

    x = x + self.attn(self.norm1(x))
    x = x + self.mlp(self.norm2(x))
    return x


class TransformerLayerWithAdaLN(nn.Module):
  def __init__(self,
               dim: int,
               heads: int,
               max_seq_len: int = 128):
    super().__init__()
    self.dim = dim
    self.heads = heads

    self.adaln1= AdaLN(dim)
    self.attn = MultiHeadAttention(dim=dim,
                                   heads=heads,
                                   max_seq_len=max_seq_len)
    self.adaln2 = AdaLN(dim)
    self.mlp = Mlp(dim=dim)

  def forward(self, x: Float[Tensor, "B S D"], cond: Float[Tensor, "B S"]):

    x_orig = x
    x, attn_gate = self.adaln1(x, cond)
    x = x_orig + attn_gate * self.attn(x)

    x_orig = x
    x, mlp_gate = self.adaln2(x, cond)
    x = x_orig + mlp_gate * self.mlp(x)

    return x


#### Tests!

We use the cell magic `%%ipytest` (which has to be placed at the top of the cell) to unit test the layer definitions.

In [ ]:
%%ipytest

def test_linear_pachification():
  pachification = LinearPachification(dim=128, patch_size=2, channels=3)
  x = torch.randn(16, 3, 32, 32)
  y = pachification(x)
  assert y.shape == (16, 256, 128)


def test_final_layer():
  pachification = LinearPachification(dim=128, patch_size=2, channels=3)
  x = torch.randn(16, 3, 32, 32)
  y = pachification(x)

  final_layer = FinalLayer(dim=128, patch_size=2, out_channels=3, out_height=32, out_width=32)

  b = y.shape[0]
  d = y.shape[-1]
  cond = torch.randn(b, d)

  z = final_layer(y, cond)
  assert z.shape == x.shape


def test_adaln():
  dim = 64
  adaln = AdaLN(dim=dim)
  assert torch.all(adaln.adaln_linear.weight == 0)
  x = torch.randn((8, 32, dim))
  cond = torch.randn((8, dim))
  out, gate = adaln(x, cond)
  assert out.shape == x.shape

  y = gate * out  # just to ensure gate and x's shape can match


def test_transformer_layer_with_adaln():
  dim = 64
  heads = 4
  layer = TransformerLayerWithAdaLN(dim=dim, heads=heads)
  assert torch.all(layer.adaln1.adaln_linear.weight == 0)
  assert torch.all(layer.adaln2.adaln_linear.weight == 0)

  x = torch.randn((8, 32, dim))
  cond = torch.randn((8, dim))
  out = layer(x, cond)
  assert out.shape == x.shape


def test_init():
  dim = 64
  heads = 4
  layer = TransformerLayerWithAdaLN(dim=dim, heads=heads)
  assert torch.all(layer.adaln1.adaln_linear.weight == 0)
  assert torch.all(layer.adaln2.adaln_linear.weight == 0)

  def custom_init(module):
    if isinstance(module, nn.Linear):
      nn.init.normal_(module.weight, 0.02)
      if module.bias is not None:
        nn.init.zeros_(module.bias)
    elif isinstance(module, AdaLN):
      # A.apply(custom_init) recursively applies custom_init to each sub-module
      # of A in a bottom-up fashion. That means, we can call a parent module's
      # reset function to override what custom_init has done to the parent's children.
      # In this case, custom_init sets each nn.Linear's weight to N(0, 0.02) Gaussian.
      # AdaLN has an nn.Linear as a sub-module and as such AdaLN's nn.Linear is also
      # initialized as a Gaussian which is, however, not what we want.
      # What we want instead is to set AdaLN's nn.Linear to zeros.
      module.reset_parameters()

  layer.apply(custom_init)

  assert torch.all(layer.adaln1.adaln_linear.weight == 0)
  assert torch.all(layer.adaln2.adaln_linear.weight == 0)


import matplotlib.pyplot as plt
def test_sinusoidal_embeddings():
  n_seq = 100
  d_model = 64
  emb = get_sinusoidal_embeddings(n_seq, d_model)
  assert emb.shape == (n_seq, d_model)
  plt.imshow(emb)


def test_sinusoidal_embeddings_2d():
  h, w = 32, 64
  d = 128
  feat = get_sinusoidal_embeddings_2d(h, w, d)  # [h*w, d]
  for i in range(h):
    for j in range(w):
      assert torch.all(feat[i * w, :d//2] == feat[i * w + j, :d//2])


  for j in range(w):
    for i in range(h):
      assert torch.all(feat[j, d//2:] == feat[w * i + j, d//2:])




## (2D) RoPE

studied in detail in https://colab.research.google.com/drive/1_g5D7Sv31Ucre6McUeYrUotzmVEoGIYj?usp=sharing

In [ ]:
from einops import rearrange

class RoPE(nn.Module):
  def __init__(self, seq_len: int, dim: int, base: int = 10_000):
    super().__init__()
    assert dim % 2 == 0
    half_dim = dim // 2
    self.half_dim = half_dim
    freq = torch.exp(-math.log(base) * torch.arange(half_dim) / half_dim) # (D)
    idx = torch.arange(seq_len) # (S)
    th = einsum(idx, freq, "S, D -> S D")
    self.register_buffer("theta", th)
    self.register_buffer("cos_th", torch.cos(th)) # [S, D//2]
    self.register_buffer("sin_th", torch.sin(th)) # [S, D//2]

  def forward(self, x: Float[Tensor, "B S D"]) -> Float[Tensor, "B S D"]:
    cos_1st_half = einsum(self.cos_th, x[..., :self.half_dim], "S D, B S D -> B S D")  # [B, S, D//2]
    sin_1st_half = einsum(self.sin_th, x[..., :self.half_dim], "S D, B S D -> B S D")  # [B, S, D//2]
    cos_2nd_half = einsum(self.cos_th, x[..., self.half_dim:], "S D, B S D -> B S D")  # [B, S, D//2]
    sin_2nd_half = einsum(self.sin_th, x[..., self.half_dim:], "S D, B S D -> B S D")  # [B, S, D//2]
    y = torch.zeros_like(x)
    y[..., :self.half_dim] = cos_1st_half - sin_2nd_half
    y[..., self.half_dim:] = sin_1st_half + cos_2nd_half
    return y


def test_rope():
  bs = 8
  seq_len = 512
  dim = 256
  rope = RoPE(seq_len, dim)
  x = torch.randn((bs, seq_len, dim))
  y = rope(x)

test_rope()


class RoPE2D(nn.Module):
  def __init__(self, h: int, w: int, dim: int, base: int = 10_000):
    super().__init__()
    assert dim % 2 == 0
    self.h = h
    self.w = w
    self.dim = dim
    self.half_dim = dim // 2
    self.rope_h = RoPE(seq_len=h, dim=self.half_dim, base=base)
    self.rope_w = RoPE(seq_len=w, dim=self.half_dim, base=base)

  def forward(self, x: Float[Tensor, "B H W D"]) -> Float[Tensor, "B H W D"]:
    b, h, w, d = x.shape
    assert (h, w, d) == (self.h, self.w, self.dim)

    y_h = rearrange(
        self.rope_h(rearrange(x[..., :self.half_dim], "B H W D -> (B W) H D")),
        "(B W) H D -> B H W D", B=b)

    y_w = rearrange(
        self.rope_w(rearrange(x[..., self.half_dim:], "B H W D -> (B H) W D")),
        "(B H) W D -> B H W D", B=b)

    y = torch.cat((y_h, y_w), dim=-1)
    return y


def test_rope2d():
  bs = 8
  h = 256
  w = 512
  dim = 128
  rope2d = RoPE2D(h, w, dim)
  x = torch.randn((bs, h, w, dim))
  y = rope2d(x)

test_rope2d()

### Transformer

Put the layer definitions together!

###

In [ ]:
from collections import defaultdict
from jaxtyping import Bool

class Transformer(nn.Module):
  def __init__(self,
               dim: int,
               num_layers: int,
               heads: int,
               patch_size: int = 2,
               channels: int = 3,
               height: int = 32,
               width: int = 32,
               num_classes: int = 10,
               num_timesteps: int = 1000,
               timestep_dim: int = 1024,
               use_2d_rope: bool=False):
    super().__init__()
    assert dim % heads == 0

    # network shape
    self.dim = dim
    self.heads = heads
    self.num_layers = num_layers

    # image shape
    self.channels = channels
    self.height = height
    self.width = width

    # pachification layer
    self.pachification = LinearPachification(
        dim=dim,
        patch_size=patch_size,
        channels=channels
    )
    # class label's embedding layer
    self.class_embedding = nn.Embedding(num_embeddings=num_classes, embedding_dim=dim)

    # timestep's pre-computes sinusoidal embedding
    self.register_buffer(
        "timestep_embedding",
        get_sinusoidal_embeddings(n_seq=num_timesteps, d_model=timestep_dim))  # B D
    # timestep's MLP projection layer
    self.timestep_proj = nn.Sequential(
        nn.Linear(in_features=timestep_dim, out_features=4 * dim, bias=False),
        nn.GELU(),
        nn.Linear(in_features=4 * dim, out_features=dim, bias=False))

    # position embedding
    self.hp = height // patch_size
    self.wp = width // patch_size
    num_visual_tokens = self.hp * self.wp
    num_tokens = num_visual_tokens
    # TODO: 2D RoPE probably works even better
    if not use_2d_rope:
      logger.info("Using absolute position embedding")
      self.register_buffer(
          "position_embedding",
          # get_sinusoidal_embeddings(n_seq=num_tokens, d_model=dim), # 1D PE
          # 2D PE
          get_sinusoidal_embeddings_2d(
              height = height // patch_size,
              width=width // patch_size,
              d_model=dim)
      ) # B D
    else:
      logger.info("Using 2D RoPE")
      self.rope2d = RoPE2D(h=self.hp, w=self.wp, dim=dim)
    self.use_2d_rope = use_2d_rope

    # transformer layers
    self.layers = nn.ModuleList()
    for i in range(num_layers):
      self.layers.append(
          TransformerLayerWithAdaLN(
              dim=dim,
              heads=heads,
              max_seq_len=num_tokens
          )
      )

    # final layer
    self.final_layer = FinalLayer(
        dim=dim,
        patch_size=patch_size,
        out_channels=channels,
        out_height=height,
        out_width=width
    )

    # initialize weights
    self.initialized_modules = defaultdict(int)
    self.apply(self._init_weights)
    logger.info(f"Layers initialized by type:\n{json.dumps(dict(self.initialized_modules), indent=2)}")

    # check
    for name, params in self.named_parameters():
      if "adaln_linear" in name:
        assert torch.all(params == 0), "AdaLN module's Linear layer (name: adaln_linear) should be zero-initialized"
      if "final_layer" in name and "linear" in name:
        assert torch.all(params == 0), "FinalLayer's Linear layer should be zero-initialized"

  def _init_weights(self, module):
    if isinstance(module, nn.Linear):
      nn.init.normal_(module.weight, mean=0.0, std=0.02)
      if module.bias is not None:
        nn.init.zeros_(module.bias)
      self.initialized_modules["linear"] += 1
    elif isinstance(module, nn.Embedding):
      nn.init.normal_(module.weight, mean=0.0, std=0.02)
      self.initialized_modules["embedding"] += 1
    elif isinstance(module, nn.RMSNorm):
      nn.init.ones_(module.weight)
      assert not hasattr(module, "bias")
      self.initialized_modules["rmsnorm"] += 1
    elif isinstance(module, nn.LayerNorm):
      if module.weight is not None:
        # This could happen because we use LayerNorm in a non-learnable
        # fashion inside AdaLN!
        nn.init.ones_(module.weight)
      if module.bias is not None:
        nn.init.zeros_(module.bias)
      self.initialized_modules["layernorm"] += 1
    elif isinstance(module, AdaLN):
      module.reset_parameters()
      self.initialized_modules["adaln"] += 1
    elif isinstance(module, FinalLayer):
      module.reset_parameters()
      self.initialized_modules["finallayer"] += 1

  def forward(self,
              x: Float[Tensor, "B C H W"],
              t: Int[Tensor, "B"],
              c: Int[Tensor, "B"],
              drop_c: Bool[Tensor, "B"] | None = None) -> Float[Tensor, "B S D"]:
    # construct the sequence
    x = self.pachification(x) # B S D
    assert x.shape[1] == self.hp * self.wp

    c_emb = self.class_embedding(c) # B D
    if drop_c is not None:
      c_emb.masked_fill_(drop_c[:, None], 0) # zero out to drop the condition

    t_emb = self.timestep_proj(self.timestep_embedding[t]) # B D
    cond = c_emb + t_emb  # B D

    # apply position embedding
    batch_size, seq_len = x.shape[:2]

    if not self.use_2d_rope:
      idx = torch.stack([torch.arange(seq_len, device=x.device)] * batch_size)  # B S
      x = x + self.position_embedding[idx]

    # pass the sequence through transformer layers
    for layer in self.layers:
      if self.use_2d_rope:
        x = rearrange(
            self.rope2d(rearrange(x, "B (H W) D -> B H W D", H=self.hp)),
            "B H W D -> B (H W) D")
      x = layer(x, cond)

    # final layer!
    x = self.final_layer(x, cond)

    return x


#### Test if the transformer works

In [ ]:
%%ipytest

def test_transformer():
  for use_2d_rope in [True, False]:
    model = Transformer(dim=128, num_layers=4, heads=4, patch_size=2,
                channels=3, height=32, width=32, num_classes=10, num_timesteps=1000,
                        use_2d_rope=use_2d_rope)

    x = torch.randn((8, 3, 32, 32))
    t = torch.randint(low=0, high=1000, size=(8, ))
    c = torch.randint(low=0, high=10, size=(8,))
    y = model(x, t, c)
    assert y.shape == x.shape

    z = model(x, t, c, drop_c=torch.ones((8,), dtype=torch.bool))
    assert z.shape == x.shape

## Sampling utilities

In [ ]:
def sample_logit_normal_timesteps(batch_size: int, num_timesteps: int, rng: torch.Generator) -> Int[Tensor, "B"]:
  """ Sample timesteps from a logit-normal distribution that is, the logit of the samples from the
  distribution follow the Gaussian/normal distribution.
  """
  # sample from the log-normal distribution
  logits = torch.randn((batch_size,), generator=rng)
  # ... apply sigmoid to the timesteps sampled from Gaussian so that the
  # log of the timesteps follow the Gaussian -- thus the name log-normal
  normalized_timesteps = torch.clip(torch.sigmoid(logits), min=0, max=1.0)
  scaled_timesteps = torch.clip(
      (num_timesteps * normalized_timesteps).to(torch.long),
      0, num_timesteps)
  return scaled_timesteps


def sample_gaussian_noise_like(x: Float[Tensor, "B C H W"], rng: torch.Generator) -> Float[Tensor, "B C H W"]:
  return torch.randn(size=x.shape, generator=rng)


In [ ]:
%%ipytest

import matplotlib.pyplot as plt


def test_sample_logit_normal_timesteps():
  bs = 10_000
  t = sample_logit_normal_timesteps(batch_size=bs, num_timesteps=1_000, rng=torch.Generator(device="cpu").manual_seed(42))
  assert t.shape == (bs, )

  plt.hist(t.numpy(), bins=100)
  plt.show()


def test_sample_gaussian_noise_like():
  x = torch.randn((8, 3, 32, 32))
  noise = sample_gaussian_noise_like(x, rng=torch.Generator(device="cpu").manual_seed(42))
  assert noise.shape == x.shape


### Random Number Generator (RNG) Manager

- We want to ensure reproducibility (and proper resuming of training)
- A large part of diffusion/flow matching model involves random number generation (e.g., sampling Gaussian noise, sampling timesteps, etc.)
- To ensure reproducibility, we need to save and restore the random number generators' internal state
- The RngManager does this

In [ ]:
class RngManager:
  def __init__(self,
               seed: int = None,
               timestep_rng_state: Tensor | None = None,
               noise_rng_state: Tensor | None = None,
               dropout_state: Tensor | None = None):
    if seed is not None:
      assert timestep_rng_state is None and noise_rng_state is None, "Cannot use both seed and rng state!!!"
      self.timestep_rng = torch.Generator(device="cpu").manual_seed(seed)
      self.noise_rng = torch.Generator(device="cpu").manual_seed(seed)
      self.dropout_rng = torch.Generator(device="cpu").manual_seed(seed)
    else:
      assert timestep_rng_state is not None and noise_rng_state is not None, "Must use either seed or rng state!!!"
      self.timestep_rng = torch.Generator(device="cpu").set_state(timestep_rng_state)
      self.noise_rng = torch.Generator(device="cpu").set_state(noise_rng_state)
      self.dropout_rng = torch.Generator(device="cpu").set_state(dropout_state)

  def serialize_state(self) -> Dict[str, Tensor]:
    return {
        "timestep_rng_state": self.timestep_rng.get_state(),
        "noise_rng_state": self.noise_rng.get_state(),
        "dropout_rng_state": self.dropout_rng.get_state(),
    }

  @classmethod
  def create_rng_manager_from_state(cls, state: Dict[str, Tensor]) -> "RngManager":
    return cls(
        seed=None,
        timestep_rng_state=state["timestep_rng_state"],
        noise_rng_state=state["noise_rng_state"],
        dropout_state=state["dropout_rng_state"],
    )



In [ ]:
%%ipytest

def test_rng_manager():
  r = RngManager(seed=42)
  torch.save({"rng_manager_state": r.serialize_state()}, "test.pt")
  rng_state = torch.load("test.pt")["rng_manager_state"]
  rr = RngManager.create_rng_manager_from_state(rng_state)

  assert torch.all(r.timestep_rng.get_state() == rr.timestep_rng.get_state())
  assert torch.all(r.noise_rng.get_state() == rr.noise_rng.get_state())

## Training
**How to train?**
- Just run the cell below
  - Make sure `TRAIN_MODE` is set to `True`

**How to run inference?**
- You can run training first as instructed above, and stop training any time you want, and then run the inference cell (next next cell).
- Or, you set `TRAIN_MODE` to `False` in the cell below and run it -- this will load a checkpoint to the model, and then run the inference cell (next next cell).






In [ ]:
%env LOGURU_LEVEL=INFO

RELEASE_COMPUTE_AFTER_TRAINIGN: bool = False
DEBUG_TIMESTEPS: bool = False

from google.colab import runtime
from typing import Optional
from dataclasses import dataclass, asdict, field
import sys
import os
import datetime
from zoneinfo import ZoneInfo
from torch.optim import AdamW
from torch.utils.data import ConcatDataset
import wandb
import matplotlib.pyplot as plt

logger.remove()
logger.add(sys.stderr, level=os.environ["LOGURU_LEVEL"])

def get_datetime_string():
  # Get the current datetime object
  now = datetime.datetime.now(tz=ZoneInfo("America/Los_Angeles"))

  # Format the datetime object as YYYYMMDD-HHMM
  formatted_time = now.strftime("%Y%m%d-%H%M")
  return formatted_time


@dataclass
class TrainerConfig:
  seed: int = 42
  batch_size: int = 96 # -> 128 for prod run
  num_epochs: int = 200 # !!
  lr: float = 1e-4
  beta1: float = 0.9
  beta2: float = 0.999
  eps: float = 1e-8
  weight_decay: float = 0.01
  # checkpointing
  ckpt_every_n_steps: int = 500
  log_every_n_steps: int = 10
  # conditioning
  dropout_cond_rate: float = 0.2

@dataclass
class DataConfig:
  num_classes: int = 10
  channels: int = 3
  height: int = 32
  width: int = 32

@dataclass
class ModelConfig:
  dim: int = 384
  num_layers: int = 12
  heads: int = 6
  patch_size: int = 2
  num_timesteps: int = 1000
  use_2d_rope: bool = False


@dataclass
class Config:
  trainer_config: TrainerConfig = field(default_factory=TrainerConfig)
  data_config: DataConfig = field(default_factory=DataConfig)
  model_config: ModelConfig = field(default_factory=ModelConfig)

  # wandb
  wandb_run_id: Optional[str] = None

  # checkpointing
  ckpt_dir: Path = workspace / "checkpoints"/ "default-image-gen-cifar10-ckpts"
  restore_from: Optional[Path] =  None
  """
  ########################################
  # Description: Debug AdaLN
  ########################################
  ckpt_dir = workspace / "checkpoints"/ "debug-adaln-image-gen-cifar10-ckpts"
  restore_from = ckpt_dir / "ckpt-global_step=00010000.pt"

  ########################################
  # Description: Debug Condition dropout
  ########################################
  ckpt_dir = workspace / "checkpoints"/ "debug-cond-image-gen-cifar10-ckpts"
  restore_from = ckpt_dir / "ckpt-global_step=00006500.pt"

  ########################################
  # Description: Debug arch tweak (GELU in MLP, SiLU in AdaLN, scale + shift + gate in AdaLN, MLP for timestep proj)
  # The one trained below is probably the **best working version**, and the checkpoint @ 47500 is the latest checkpoint.
  # It reaches about half of the target training steps (assuming #epoch is 200 in trainer_config as shown below)
  ########################################
  ckpt_dir = workspace / "checkpoints"/ "debug-arch-image-gen-cifar10-ckpts"
  restore_from =  ckpt_dir /  "ckpt-global_step=00047500.pt"
  """

  def __post_init__(self):
    assert self.ckpt_dir is not None
    if not self.ckpt_dir.exists():
      self.ckpt_dir.mkdir(parents=True, exist_ok=True)
      logger.info(f"Checkpoint dir created @ {self.ckpt_dir}")

# set up device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"{device=}")


def inference(
    model: nn.Module,
    model_config: ModelConfig,
    data_config: DataConfig,
    classes: List[str],
    num_inference_steps: int = 50,
    samples_per_class: int = 8,
    cfg: float = 4.5,
    hide_pbar: bool = False # if set, hide the progress bar -- useful during training to un-clutter stdio
):
  model.eval()
  num_classes = len(classes)


  test_rng_manager = RngManager(seed=42)
  x = sample_gaussian_noise_like(
      x=torch.empty((num_classes * samples_per_class,
                    data_config.channels,
                    data_config.height,
                    data_config.width)),
      rng=test_rng_manager.noise_rng).to(device)

  inference_bs = x.shape[0]

  # Use linspace to ensure we go exactly from 0 to 1000, so we don't end early at 980
  discrete_timesteps = torch.linspace(0,
                                    model_config.num_timesteps,
                                    num_inference_steps + 1,
                                    dtype=torch.long).to(device)

  for idx in tqdm.tqdm(range(len(discrete_timesteps) - 1), disable=hide_pbar):
    _t_shape = torch.ones((inference_bs,), dtype=torch.long, device=device)
    curr_t = discrete_timesteps[idx] * _t_shape
    next_t = discrete_timesteps[idx + 1]  * _t_shape

    # labels has layout [0, 1, ..., 9, 0, 1, ..., 9, ..., 0, 1, ..., 9]
    labels = torch.arange(num_classes, device=device).repeat(samples_per_class)

    with torch.no_grad():
      v_cond = model(x=x, t=curr_t, c=labels)
      drop_all = torch.ones((inference_bs,), dtype=torch.bool, device=device)
      v_null = model(x=x, t=curr_t, c=labels, drop_c=drop_all)
      v = v_null + cfg * (v_cond - v_null)

    dt = (next_t - curr_t) / model_config.num_timesteps
    bc_dt = rearrange(dt, "b -> b 1 1 1")
    x = x + v * bc_dt

  x = (x + 1) * 0.5 # [0, 1]
  x = (x * 255).clip(0, 255).to(torch.uint8)

  from torchvision.utils import make_grid
  pad = 2
  grid = make_grid(x, nrow=num_classes, padding=pad, normalize=False)
  grid = rearrange(grid, "c h w -> h w c")
  grid = grid.cpu().numpy()

  fig = plt.figure(figsize=(10, 8))
  plt.imshow(grid)

  # Calculate center position for each class column to place the x-ticks
  img_width = data_config.width
  tick_positions = [pad + i * (img_width + pad) + img_width / 2 for i in range(num_classes)]
  plt.xticks(ticks=tick_positions, labels=classes, rotation=30)
  plt.yticks([])

  return fig


def save_ckpt(model, optimizer, rng_manager, global_step: int, ckpt_path: str | Path, wandb_run_id: str | None = None):
  torch.save({
      "model_state_dict": model.state_dict(),
      "optimizer_state_dict": optimizer.state_dict(),
      "global_step": global_step,
      "rng_manager_state": rng_manager.serialize_state(),
      "wandb_run_id": wandb_run_id
  }, ckpt_path)


def create_model(config: ModelConfig, data_config: DataConfig):
  model = Transformer(dim=config.dim,
                      num_layers=config.num_layers,
                      heads=config.heads,
                      patch_size=config.patch_size,
                      channels=data_config.channels,
                      height=data_config.height,
                      width=data_config.width,
                      num_classes=data_config.num_classes,
                      num_timesteps=config.num_timesteps,
                      use_2d_rope=config.use_2d_rope).to(device)
  return model

def train_model(config: Config):
  model_config: ModelConfig = config.model_config
  trainer_config: TrainerConfig = config.trainer_config
  data_config: DataConfig = config.data_config

  restore_from = config.restore_from
  ckpt_dir = config.ckpt_dir

  restored_ckpt = None
  wandb_run_id = None
  if restore_from is not None:
    logger.info(f"Restoring from {restore_from}")
    restored_ckpt = torch.load(restore_from)
    wandb_run_id = restored_ckpt.get("wandb_run_id")

  if wandb_run_id is None:
    wandb_run_id = config.wandb_run_id

  # Initialize wandb run
  wandb.init(
      project="class-to-image-cifar10",
      id=wandb_run_id,
      resume="allow",
      config={
          "model": asdict(model_config),
          "trainer": asdict(trainer_config),
          "data": asdict(data_config)
      }
  )

  # training mode
  model = create_model(config=model_config,
                       data_config=data_config)
  model.train()

  # set up optimizer
  optimizer = AdamW(model.parameters(),
                    lr=trainer_config.lr,
                    betas=(trainer_config.beta1, trainer_config.beta2),
                    eps=trainer_config.eps,
                    weight_decay=trainer_config.weight_decay)

  if restored_ckpt is not None:
    model.load_state_dict(restored_ckpt["model_state_dict"])
    optimizer.load_state_dict(restored_ckpt["optimizer_state_dict"])


  # set up dataloader
  transform_train = transforms.Compose([
      # transforms.RandomCrop(32, padding=4),
      transforms.RandomHorizontalFlip(),
      transforms.ToTensor(),
      transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # normalize to the range of [-1, 1]
  ])


  # Load the CIFAR10 training dataset
  trainset = torchvision.datasets.CIFAR10(
      root=data_dir, train=True, download=True, transform=transform_train)
  trainset = ConcatDataset([trainset] * trainer_config.num_epochs)


  # Create a DataLoader for the training set
  trainloader = DataLoader(
      trainset,
      batch_size=trainer_config.batch_size,
      shuffle=True,
      num_workers=2,
      collate_fn=my_collate_fn,
      generator=torch.Generator(device="cpu").manual_seed(trainer_config.seed))


  if restored_ckpt is None:
    rng_manager = RngManager(seed=trainer_config.seed)
    global_step = 0
  else:
    rng_manager = RngManager.create_rng_manager_from_state(restored_ckpt["rng_manager_state"])
    global_step = restored_ckpt["global_step"]


  collected_timesteps = None
  if DEBUG_TIMESTEPS:
    collected_timesteps = []

  pbar = tqdm.tqdm(trainloader, unit="batch", initial=global_step)
  for batch in pbar:
    images = batch["images"].to(device)
    labels = batch["labels"].to(device)

    # sample timesteps
    timesteps = sample_logit_normal_timesteps(
        batch_size=images.shape[0],
        num_timesteps=model_config.num_timesteps,
        rng=rng_manager.timestep_rng).to(device)  # [B]
    # sample noises
    noises = sample_gaussian_noise_like(x=images, rng=rng_manager.noise_rng).to(device) # [B C H W]
    # construct the noisy image where at timestep 0, we have pure noise (at inference time, we solve the ODE dx/dt = v from time 0 to time 1 by )
    # alpha is the normalized (and broadcasted timesteps)
    alpha = rearrange(timesteps / model_config.num_timesteps, "b -> b 1 1 1")
    noisy_images = (1 - alpha) * noises + alpha * images
    velocity = images - noises

    # randomly dropout some conditions
    drop_c = (torch.rand((trainer_config.batch_size,),
                         generator=rng_manager.dropout_rng) < trainer_config.dropout_cond_rate).to(device)


    # DEBUG
    # logger.debug(f"{drop_c=}")
    # logger.debug(f"{timesteps=}")
    # logger.debug(f"{labels=}")
    if collected_timesteps is not None:
      # collect the sampled timesteps to inspect its distribution later
      collected_timesteps.append(timesteps.cpu())

    pred = model(x=noisy_images, t=timesteps, c=labels, drop_c=drop_c)

    loss = F.mse_loss(pred, velocity)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    wandb.log({"train/loss": loss.item()}, step=global_step)
    pbar.set_postfix({"loss": f"{loss.item():0.4f}"})

    global_step += 1
    current_datetime = get_datetime_string()

    if global_step % trainer_config.ckpt_every_n_steps == 0:
      ckpt_path = ckpt_dir / f"ckpt-{global_step=:08d}.pt"
      save_ckpt(model, optimizer, rng_manager, global_step, ckpt_path, wandb.run.id if wandb.run else None)
      logger.debug(f"Saved checkpoint to {ckpt_path}")

      model.eval()
      fig = inference(model, model_config, data_config, classes, hide_pbar=True)
      if wandb.run is not None:
        wandb.log({"inference/samples": wandb.Image(fig)}, step=global_step)
      # plt.show()
      # plt.close(fig)
      model.train()

    if global_step % trainer_config.log_every_n_steps == 0:
      with open(ckpt_dir / "loss.txt", "a") as f:
        f.write(f"global_step: {global_step:08d}, datetime: {current_datetime}, loss: {loss.item():0.4f}\n")
  try:
      ckpt_path = ckpt_dir / f"ckpt-{global_step=:08d}.pt"
      save_ckpt(model, optimizer, rng_manager, global_step, ckpt_path, wandb.run.id if wandb.run else None)
      logger.info(f"Saved final checkpoint to {ckpt_path}")
  except Exception as e:
    logger.error(f"Error saving checkpoint: {e}")
  finally:
    wandb.finish()
    if RELEASE_COMPUTE_AFTER_TRAINIGN:
      logger.info("Release compute ...")
      runtime.unassign()


def test_model(config: Config):
  model_config: ModelConfig = config.model_config
  data_config: DataConfig = config.data_config
  restore_from = config.restore_from
  ckpt_dir = config.ckpt_dir

  # inference mode
  model = create_model(config=model_config,
                       data_config=data_config)
  model.eval()

  assert restore_from is not None, "Restore from must be specified in inference mode!!"
  logger.info(f"Restoring from {restore_from}")
  restored_ckpt = torch.load(restore_from)
  model.load_state_dict(restored_ckpt["model_state_dict"])

  fig = inference(model, model_config, data_config, classes, cfg=1.0)
  plt.show()
  plt.close(fig)


def main():
  # instantiate configs -- modify the configs as needed
  experiment_name =  "enhance-quality-image-gen-cifar10-ckpts-rope2d"
  experiment_name += "-test-1"

  ckpt_dir = workspace / "checkpoints"/ experiment_name
  restore_from = ckpt_dir / "ckpt-global_step=00005500.pt"

  config = Config(
    trainer_config=TrainerConfig(
        ckpt_every_n_steps=500,
        batch_size=512,
        num_epochs=400,
    ),
    data_config=DataConfig(),
    model_config=ModelConfig(
        use_2d_rope=True
    ),
    ckpt_dir=ckpt_dir,
    restore_from=restore_from,
    wandb_run_id=experiment_name
  )

  is_train = True
  if is_train:
    train_model(config)
  else:
    test_model(config)


main()


Release memory if needed to work around Out-of-Memory (OOM) issues.

In [ ]:
# import gc
# import torch

# # Force garbage collection
# gc.collect()

# # Empty PyTorch's CUDA cache
# torch.cuda.empty_cache()
# print("GPU memory safely cleared!")
